# 01 — Model Comparison (STT / Translation / TTS)

**Purpose:** empirically decide which free-tier STT, Translation, and TTS options to commit to
for the Nepali ↔ English MVP, instead of guessing.

This notebook is meant to be run once (or whenever a new candidate model is worth checking),
and the conclusion should be written back into the README / NOTES.md.

**What we're comparing:**
- STT: `faster-whisper` model sizes (`tiny`, `base`, `small`) on a Nepali sample
- Translation: `deep-translator` (Google) output quality on a few tricky sentences
- TTS: `gTTS` Nepali voice quality / naturalness (listen and judge manually)

> Add a real Nepali audio sample at `../assets/sample_ne.wav` before running the STT cells.


# Setup


In [12]:
import sys
sys.path.append("..")  # so we can import from src/

import time
from src.translator import translate_ne_to_en, translate_en_to_ne,translate_text
from src.text_to_speech import synthesize_speech
from IPython.display import Audio, display


## 1. STT model size comparison

Compares transcription speed and output across Whisper model sizes.
Swap in your own Nepali audio clip at `SAMPLE_AUDIO` below.


In [9]:
SAMPLE_AUDIO = "../src/nepali_speech.mp3"  # replace with a real Nepali clip

from faster_whisper import WhisperModel

results = {}

for size in ["tiny", "base", "small","medium"]:
    print(f"--- Loading model: {size} ---")
    model = WhisperModel(size, device="cpu", compute_type="int8")

    start = time.time()
    segments, info = model.transcribe(SAMPLE_AUDIO, language="ne", beam_size=5)
    text = " ".join(s.text.strip() for s in segments)
    elapsed = time.time() - start

    results[size] = {"text": text, "seconds": round(elapsed, 2)}
    print(f"[{size}] ({elapsed:.2f}s): {text}\n")

results


--- Loading model: tiny ---
[tiny] (0.49s): Namaste, mirror Nam Ayus Ho, TV Goha Bata Ho.

--- Loading model: base ---
[base] (0.85s): Namaste, mereu nam aayu suho, timi guha bata hoo.

--- Loading model: small ---
[small] (2.81s): नमस्ते मेरो नाम आयूस हो तिमि को हा बात हो

--- Loading model: medium ---
[medium] (6.78s): नमस्ते, मेरो नाम आयुश हो, तिमी गहा बाता हो



{'tiny': {'text': 'Namaste, mirror Nam Ayus Ho, TV Goha Bata Ho.',
  'seconds': 0.49},
 'base': {'text': 'Namaste, mereu nam aayu suho, timi guha bata hoo.',
  'seconds': 0.85},
 'small': {'text': 'नमस्ते मेरो नाम आयूस हो तिमि को हा बात हो',
  'seconds': 2.81},
 'medium': {'text': 'नमस्ते, मेरो नाम आयुश हो, तिमी गहा बाता हो',
  'seconds': 6.78}}

**Judgment call:** pick the smallest model whose transcript is still accurate enough.
Write the decision (and why) into `NOTES.md` or the README once you've compared outputs above.


## 2. Translation quality check

A few sentences that are easy to get subtly wrong (idioms, ambiguous words, formal/informal register).
Read each translation and judge manually — no automatic metric yet (that comes later, in
`03_evaluation.ipynb`, once we have a reference translation set).


In [14]:
test_sentences_ne = [
    "मलाई पोखरा जानको लागि बस टिकट चाहिन्छ।",
    "तपाईंलाई कस्तो छ?",
    "म बिरामी छु, मलाई डाक्टर चाहिन्छ।",
    "धन्यवाद, तपाईंको सहयोगको लागि।",
]

for s in test_sentences_ne:
    try:
        translated = translate_text(s, source_lang="ne", target_lang="en")
        print(f"NE: {s}\nEN: {translated}\n")
    except Exception as e:
        print(f"Error translating '{s}': {e}")


NE: मलाई पोखरा जानको लागि बस टिकट चाहिन्छ।
EN: I need a bus ticket to go to Pokhara.

NE: तपाईंलाई कस्तो छ?
EN: how are you

Error translating 'म बिरामी छु, मलाई डाक्टर चाहिन्छ।': म बिरामी छु, मलाई डाक्टर चाहिन्छ। --> No translation was found using the current translator. Try another translator?
NE: धन्यवाद, तपाईंको सहयोगको लागि।
EN: Thank you for your help.



In [24]:
from deep_translator import GoogleTranslator

test_sentences_en = [
    "Where is the nearest hospital?",
    "Can you help me find a hotel?",
    "I would like to buy a bus ticket to Pokhara.",
]

for s in test_sentences_en:
    try:
        translated = GoogleTranslator(source="en", target="ne").translate(s)
        print(f"EN: {s}\nNE: {translated}\n")
    except Exception as e:
        print(f"Error translating '{s}': {e}")


Error translating 'Where is the nearest hospital?': Where is the nearest hospital? --> No translation was found using the current translator. Try another translator?
Error translating 'Can you help me find a hotel?': Can you help me find a hotel? --> No translation was found using the current translator. Try another translator?
EN: I would like to buy a bus ticket to Pokhara.
NE: म पोखरा जाने बसको टिकट किन्न चाहन्छु।

